# Просмотр speech-данных
Запускать на dev-1 в доступной notebook-среде; отдельный сервер не требуется.
Зависимости ядра: boto3, duckdb, pandas, IPython. Используйте read-only S3 credentials
через стандартные AWS-переменные окружения (или AWS profile), `S3_ENDPOINT` и
`AWS_CA_BUNDLE`. Не вставляйте ключи в notebook. Данные читаются из существующего MinIO.

Снимок предназначен для инспекции, не является обучающим релизом.
`not_linked_or_not_available` означает, что исходная метка пока не привязана или недоступна.
Ответ Gemini не означает ручную проверку. После работы очистите outputs перед сохранением в Git.


In [ ]:
import os, json, hashlib, tempfile
from pathlib import Path
from urllib.parse import urlparse
import boto3
import duckdb
from IPython.display import Audio, display

s3 = boto3.client('s3', endpoint_url=os.environ['S3_ENDPOINT'])
bucket = os.environ.get('S3_DATASET_BUCKET', 'speech-datasets')
snapshot = json.loads(s3.get_object(Bucket=bucket, Key='analytics/latest.json')['Body'].read())
assert snapshot['purpose'] == 'inspection_only_not_training_release'
print('Снимок:', snapshot['snapshot_at'])
print({name: table['rows'] for name, table in snapshot['tables'].items()})
cache = tempfile.TemporaryDirectory(prefix='speech-inspect-')
db = duckdb.connect()

def load_table(name):
    # Только файлы одного manifest: glob по всем версиям удвоил бы результаты.
    assert name in snapshot['tables']
    meta = snapshot['tables'][name]
    path = Path(cache.name) / (name + '.parquet')
    if not path.exists():
        s3.download_file(snapshot['bucket'], meta['key'], str(path))
        with path.open('rb') as f:
            assert hashlib.file_digest(f, 'sha256').hexdigest() == meta['sha256']
    db.read_parquet(str(path)).create_view(name, replace=True)

for name in ('sources', 'progress', 'examples', 'passes', 'publications'):
    load_table(name)


## Сводка и исходники, включая ещё не обработанные

In [ ]:
display(db.sql("SELECT corpus, preparation_state, count(*) AS files, sum(bytes) AS bytes FROM sources GROUP BY ALL ORDER BY corpus, preparation_state").df())
display(db.sql("SELECT * FROM progress ORDER BY corpus, prompt_variant, processing_state").df())


In [ ]:
corpus_filter = None  # Например: имя корпуса из сводки.
display(db.execute("SELECT * FROM sources WHERE (? IS NULL OR corpus=?) ORDER BY object_key LIMIT 50", [corpus_filter, corpus_filter]).df())


## Сравнение исходной разметки и проходов Gemini

In [ ]:
examples = db.execute("SELECT * FROM examples WHERE (? IS NULL OR corpus=?) ORDER BY flagged_passes DESC, updated_at DESC LIMIT 50", [corpus_filter, corpus_filter]).df()
display(examples)


In [ ]:
# Выберите номер строки таблицы выше.
example_index = 0
if examples.empty:
    print('Примеры пока отсутствуют для выбранного корпуса.')
else:
    selected = examples.iloc[example_index]
    passes = db.execute("SELECT * FROM passes WHERE batch_id IS NOT DISTINCT FROM ? AND sample_id=? ORDER BY prompt_variant", [selected['batch_id'], selected['sample_id']]).df()
    display(passes[['prompt_variant', 'processing_state', 'error_code', 'original_annotation', 'gemini_text', 'model', 'contract_sha256']])
    audio = s3.get_object(Bucket=selected['audio_bucket'], Key=selected['audio_key'])['Body'].read()
    assert hashlib.sha256(audio).hexdigest() == selected['audio_sha256']
    display(Audio(audio))
    provenance = json.loads(passes.iloc[0]['source_provenance'] or '{}')
    print('Исходник и границы фрагмента:')
    display(provenance)


## Исходный объект
Исходник может быть аудио, большим архивом или Parquet с вложенным аудио.
`source_provenance` показывает его точное расположение и элемент (`member`).
Следующая функция выводит метаданные без скачивания многогигабайтного архива.
Для самостоятельного аудиофайла доступно прослушивание (лимит 50 MiB).
Для архивов/Parquet выше прослушивается подготовленный фрагмент, не весь исходный контейнер.


In [ ]:
def inspect_source(source_bucket, source_key, play=False):
    meta = s3.head_object(Bucket=source_bucket, Key=source_key)
    print({'uri': f's3://{source_bucket}/{source_key}', 'bytes': meta['ContentLength'], 'etag': meta['ETag']})
    if play:
        if Path(source_key).suffix.lower() not in {'.wav', '.mp3', '.opus', '.ogg', '.flac', '.m4a'}:
            raise ValueError('Это контейнер данных; выберите его аудиофрагмент через examples.')
        if meta['ContentLength'] > 50 * 1024**2:
            raise ValueError('Слишком большой файл для inline-плеера; используйте подготовленный фрагмент.')
        display(Audio(s3.get_object(Bucket=source_bucket, Key=source_key)['Body'].read()))

# Можно указать bucket/object_key любой строки sources, в том числе необработанной.
# inspect_source('speech-corpora', 'staging/...', play=False)
if not examples.empty and provenance.get('bucket') and provenance.get('key'):
    inspect_source(provenance['bucket'], provenance['key'])


## Опубликованные результаты
Пустая таблица означает отсутствие опубликованного релиза, а не потерю исходников или промежуточных ответов.

In [ ]:
display(db.sql('SELECT * FROM publications ORDER BY created_at DESC').df())
